In [ ]:
import torch
import yaml
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from torchvision.ops import nms
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.ops import generalized_box_iou_loss
from torch import nn
from helper.utils import CustomImageDataset

In [ ]:
torch.manual_seed(42)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
weights = ResNet50_Weights.DEFAULT # weights = IMAGENET1K_V2
preprocess = weights.transforms()

In [ ]:
model = resnet50(weights=weights) 
for param in model.parameters():
    param.requires_grad = False # freeze the parameters of the model to only train the new layers for object detection first

## Build Dataloader loading the Custom Dataset

In [ ]:
mean = preprocess.mean
std = preprocess.std

# double the input size (=> doubles output size)
resize_size = preprocess.resize_size[0] * 2
crop_size = preprocess.crop_size[0] * 2

# the test transforms are also transferred to v2-transformations, so they can be simulataneously applied to image and bounding boxes
# => with the normal weights preprocess, they are only applied to the image, so the bounding boxes are slightly offset because of the center crop
test_transforms = v2.Compose([
    v2.Resize(resize_size),
    v2.CenterCrop(crop_size),
    v2.ToDtype(torch.float32, scale=True), # also scale the image to [0, 1] from [0, 255] after loading the image
    v2.Normalize(mean=mean, std=std)
])


In [ ]:
# Prepare Datasets

image_size = crop_size

# as there are not too many images in the dataset, using data augmentation helps to increase the number of unique images preventing overfitting
train_transforms = v2.Compose([
    v2.RandomResizedCrop(image_size, (0.6, 1)),
    v2.RandomPerspective(0.75, 0.2),
    v2.ColorJitter(0.3, 0.2, 0.2, 0.1),
    v2.RandomGrayscale(0.1),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=mean, std=std)
])

dataset_path = "data/Traffic_Sign/car"

train_dataset =  CustomImageDataset(
    annotations_dir=f"{dataset_path}/train/labels",
    img_dir=f"{dataset_path}/train/images",
    transform=train_transforms
)

val_dataset =  CustomImageDataset(
    annotations_dir=f"{dataset_path}/valid/labels",
    img_dir=f"{dataset_path}/valid/images",
    transform=test_transforms
)

In [ ]:
# load class names from yaml file

with open(f"{dataset_path}/data.yaml", "r") as f:
    class_names = yaml.safe_load(f)["names"]

In [ ]:
# Prepare Dataloader and define collate function to handle the different amounts of bounding boxes in each image

batch_size = 64
num_workers = 0 # multiple workers will not work in a ipynb on macOS

def collate_fn(batch):
    return list(zip(*batch))
    # rearrange the batch into two tuples, each with len()=64: [(image_1, image_2, ...), (target_1, target_2, ...)]

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)

## Changing Architecture to Object Detection instead of Classification

In [ ]:
# setting the avg_pool and fc layer in the network to nn.Identity() does not work, because in the forward pass there is still a .flatten() which destroys the needed shape of the output tensor
# => nn.Identity() would basically does nothing and is just forwarding the input to the next layer

# so instead, we get all the elements of the network as a list => the last two layers (avg_pool and fc) are then stripped including the forward pass of those modules

model = nn.Sequential(*list(model.children())[:-2])
print(model)

In [ ]:
# the model gives an output of a tensor with shape [64, 2048, 14, 14] (when using batch size 64) => 14x14 output feature map because the input size was doubled to 448x448
# so the desired output is [64, 20, 14, 14] as we have 15 class probabilities + 4 coordinates + 1 objectness score (prob for center of the bounding box) and B = 1 => num_channels = (B*5)+C = (1*5)+15 = 20

class ObjectDetectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=2048, out_channels=2048, kernel_size=3, padding=1, bias=False)
        self.batchnorm1 = nn.BatchNorm2d(2048)
        self.conv2 = nn.Conv2d(in_channels=2048, out_channels=2048, kernel_size=3, padding=1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(2048)
        self.outputmapping = nn.Conv2d(in_channels=2048, out_channels=20, kernel_size=1, bias=True)


    def forward(self, x):
        # extra Layer 1
        x = F.relu(self.batchnorm1(self.conv1(x)))
        # extra Layer 2
        x = F.relu(self.batchnorm2(self.conv2(x)))
        # mapping on desired output tensor shape
        x = self.outputmapping(x)
        return x

model.add_module("Object Detection Head", ObjectDetectionHead())
model.to(device)

print(model)

### Goal

The goal is to make an YOLOv1 like model.
To not get lost in deep architectural changes and details to optimize for speed and scope of the model and instead focus on implementing the fundamental fine-tuning process to get from classification to
object detection, the focus will lie on the more simple implementation of YOLOv1-like architecture proposals and not go further into later improvements of the model.

Some improvements will be done tho, as e.g. using a higher resulution and changing the loss function
What exactly will be done:
1. double the input to 448x448 to get an output of 14x14 (more cells meaning less collisions when predicting multiple objects and have more detailed spatial information) => YOLOv2 style
2. adding a few more 3x3 convolution layers (to balance the more complex task)
3. map on the proposed output-tensor shape for the task ([batch_size, 20, 14, 14])
4. loss function is implemented as:
    - Sigmoid + BCE for the objectness loss => as an image can have multiple objects
    - Softmax + CE for the classification loss (only on cells with objectness = 1)
    - Sigmoid + GIoU + (MSE for w and h) for the localization loss => only GIoU is leading to an unstable training due to possibly quick convergence into close-to-zero-gradient areas (also only on cells with objectness=1)
5. using AdamW optimizer and appropriate learning rate + learning rate scheduler to optimize training convergence and stability
6. after training of the head the whole model is unfrozen and fine-tuned for a few epochs
7. (unlike YOLOv1, no fully connected layer will be implemented)

### Limitations and Explanation
YOLOv1 cannot detect multiple classes in the same cell, it can only propose two bounding boxes for ONE object which center is in that cell. Those boxes will eventually specialize during training (e.g. on different sizes and shape).

Using B > 1 makes the task easier to predict accurate boxes, as more weights are used for predicting bounding boxes and not one set of weights has to make the regression for all kinds of boxes in the dataset.
Later improvements fix that limitation partly, e.g. by adding multiple anchors which specialize on different bounding box shapes and sizes to be able to predict multiple objects per cell.

To avoid overcomplicating the task and focussing on achieving the tranfer from classification to object detection by defining a working loss function and optimizing training parameters. As the extra layers at the end are pretty large and the output is predicted from a previous 2048x14x14 map, the extra parameters to correctly perform the regression task should not be necessary to achieve satisfactory results.


## Defining Loss Function and Calculation
- Objectness Loss: Sigmoid + BCE => an image can have multiple objects
- Classification Loss: Softmax + CE (only on cells with objectness = 1)
- Localization Loss: Sigmoid + GIoU + (MSE for w and h) => only GIoU is leading to an unstable training due to quick convergence into close-to-zero-gradient areas (also only on cells with objectness = 1)

In [ ]:
# defining weights for the loss calculation
no_obj_w = 0.2 
loc_w = 4
obj_w = 12
class_w = 1
loc_l2_factor = 1

In [ ]:
# tensor shape: [batch_size, 20, 14, 14]

# feature map assignment
# (0)           objectness
# (1)           bounding box - x (relative to cell border)
# (2)           bounding box - y (relative to cell border)
# (3)           bounding box - w (relative to image size)
# (4)           bounding box - h (relative to image size)
# (5) - (19)    class prob per cell 


def get_target_maps(map_size, batch_size, y, device=None):
    # to get the correct 2D Feature map for the objectness "map" for each batch item, get the x_center and y_center (relative to image size) of all objects => multiply by image size and store in a tensor
    obj_map = torch.zeros(batch_size, map_size, map_size).to(device)
    class_maps = torch.ones(batch_size, map_size, map_size, dtype=torch.long).to(device) * -1
    loc_maps = torch.zeros(batch_size, 4, map_size, map_size).to(device)

    for b_idx, b_item in enumerate(y):
        for object in b_item:
            (c, x_cent, y_cent, w, h) = object
            x_coord, y_coord = min(13, int(x_cent * map_size)), min(13, int(y_cent * map_size)) # min(13, ...) because when x_cent or y_cent is 1.0 an OutOfBoundsError would occur (index = 14)
            x_offset, y_offset = (x_cent * map_size - x_coord), (y_cent * map_size - y_coord)

            obj_map[b_idx, y_coord, x_coord] = 1
            class_maps[b_idx, y_coord, x_coord] = int(c)
            loc_maps[b_idx, :, y_coord, x_coord] = torch.tensor([x_offset, y_offset, w, h]) # inject the id tensor along the second axis by getting all element of that pixel for all 4 feature maps
        
    return obj_map, class_maps, loc_maps


def yolo_to_bbox(yolo_box, img_size, map_size):
    # 64, 4, 14, 14
    cell_width = img_size / map_size

    # yolo_box[:, 0:2] is now only the x-offset and the y-offset feature map, but for GIoU we need the absolute coordinates of x1 and y1
    x_scale_tensor = torch.arange(0, map_size).expand(map_size, -1).to(device) # arange creates vector [0, ..., map_size-1], expand duplicates that to reach shape (map_size x map_size)
    y_scale_tensor = torch.arange(0, map_size).view((-1, 1)).expand(-1, map_size).to(device) # here we need to change the row vector to a "column vector", then expand "to the right"
    # calculate absolute x-coordinates => every column index * pixel resolution of one cell + the x-offset in pixels
    abs_x = cell_width * x_scale_tensor + yolo_box[:, 0] * cell_width 
    # calculate absolute y-coordinates => same as x-coordinate calculation but now with multiplying each row index
    abs_y = cell_width * y_scale_tensor + yolo_box[:, 1] * cell_width
    
    # also convert relative height/width into absolute height/width
    abs_w = yolo_box[:, 2] * img_size
    abs_h = yolo_box[:, 3] * img_size

    x1 = abs_x - abs_w / 2
    y1 = abs_y - abs_h / 2
    x2 = abs_x + abs_w / 2
    y2 = abs_y + abs_h / 2

    bbox_tensor = torch.stack((x1, y1, x2, y2), dim=3)
    return bbox_tensor


def objectness_loss(pred, target_obj):
    # using Sigmoid + BCE for the objectness score as there can be mulitple objects in an image (higher weight on objectness = 1 because there are more pixels with no objects than with objects)
    pred_obj = pred[:, 0] 
    loss_obj = F.binary_cross_entropy_with_logits(input=pred_obj, target=target_obj.float(), reduction="none") # this function automatically applies sigmoid function to the logits
    weight_map = torch.where(target_obj == 1, torch.ones_like(loss_obj), torch.ones_like(loss_obj) * no_obj_w) 
    loss_obj *= weight_map # multiply by the no_obj_w, where no object should be detected and by 1 where an object should be detected
    # as there are way more pixels with no objects, weight no object pixels less 
    # with standard reduction = "mean" the mean is taken over every pixel over every batch so Sum_of_BCE / (64 * 14 * 14) 
    return loss_obj.mean()


def localization_loss(pred, target_loc, img_size, map_size, target_obj):
    # using GIoU + L2 distance penalty for width and height => GIoU to normalize for different sizes of objects
    # by only using GIoU tho, the model can't confidently localize small objects, which is adressed by the L2 distance penalty => encourages for smaller boxes, helping the model get out of the small gradients plateau at the beginning of training 
    yolo_pred = torch.sigmoid(pred[:, 1:5])
    # prediction and target localization tensors have shape [64, 4, 14, 14], but for GIoU loss, shape [64, 14, 14, 4] is needed
    bbox_pred = yolo_to_bbox(yolo_pred, img_size=img_size, map_size=map_size)
    bbox_target = yolo_to_bbox(target_loc, img_size=img_size, map_size=map_size)

    all_giou_loss = generalized_box_iou_loss(bbox_pred, bbox_target)
    l2_w = loc_l2_factor * (yolo_pred[:, 2].sqrt() - target_loc[:, 2].sqrt()) ** 2
    l2_h = loc_l2_factor * (yolo_pred[:, 3].sqrt() - target_loc[:, 3].sqrt()) ** 2
    # the distance penalty (like YOLOv1 does it) is necessary, because otherwise the model does not correctly localize small images
    # when only using the GIoU-Loss, the gradient is really small with a small IoU and then oscillates when breaking out of that plateau (doesn't even get out of the plateau with normal batch sizes, like e.g. 64)
    # boxes stay large with the GT box in that pred box (GIoU equals roughly IoU, so GIoU-Loss around 1) => with the distance penalty we push the model out of that plateu by encouraging smaller width and height
    giou_l2_loss = all_giou_loss + l2_w + l2_h
    mean_relevant_giou_loss = torch.masked_select(giou_l2_loss, target_obj > 0).mean()

    return mean_relevant_giou_loss


def classification_loss(pred, target_class):
    # Softmax + Cross Entropy for the classification loss (only one object per cell can be detected, so only one class is possible) => loss is only calculated on cells with objectness = 1
    class_logits = pred[:, 5:]
    loss_class = F.cross_entropy(class_logits, target_class, reduction="mean", ignore_index=-1) # ignore all indices with no class assignnment (objectness = 0) => class = -1
    # Cross Entropy loss automaticall calculates the softmax over dim = 1 (softmax for each pixel between all feature maps) and compares with the correct class index with target_value = 1 for that class
    # with standard reduction = "mean" the mean is taken over the loss for every pixel over every batch so Sum_of_CE / (64 * 15 * n) => n is the number of pixels with objectness = 1; 15 the number of classes
    # => the 15 dimensions from the classes are reduced to one dimension (CE loss)

    return loss_class


def calc_detection_loss(pred, y, img_size=image_size, device=device):
    # calculates all three losses and weights them to account for the different orders of magnitude and importances
    map_size = pred.shape[-1]

    target_obj, target_class, target_loc = get_target_maps(map_size=map_size, batch_size=pred.shape[0], y=y, device=device)
    
    loss_obj = objectness_loss(pred, target_obj)
    loss_class = classification_loss(pred, target_class)
    loss_loc = localization_loss(pred, target_loc, img_size=img_size, map_size=map_size, target_obj=target_obj)


    loss = obj_w * loss_obj + class_w * loss_class + loc_w * loss_loc
    return loss, obj_w * loss_obj, class_w * loss_class, loc_w * loss_loc


## Test if model is learning by overfitting on single batch

In [ ]:
test_batch = next(iter(train_dataloader))

overfit_weight_decay = 1e-4
overfit_lr = 1e-3
overfit_epochs = 200
overfit_warmup_epochs = 40
overfit_optimizer = torch.optim.AdamW(model.parameters(), lr=overfit_lr, weight_decay=overfit_weight_decay) # seperat optimizer for overfitting, so seperate the internal stats (like momentum)
overfit_warmup = torch.optim.lr_scheduler.LambdaLR(overfit_optimizer, lr_lambda=lambda epoch: min(1, (epoch + 1) / overfit_warmup_epochs))
# for overfit test, just use a simple lr_scheduler with a warmup phase so the gradients don't get too large at the beginning => goal is only to see, if the model is able to memorize the batch

all_losses = []

model.train()

X, y = torch.stack(test_batch[0]).to(device), test_batch[1]

for epoch in range(overfit_epochs):
    pred = model(X)
    loss, loss_obj, loss_class, loss_loc = calc_detection_loss(pred, y)
    print("-------------------\nLoss Distribution\nObjectness Loss:", loss_obj, "\nClassification Loss:", loss_class, "\nLocalization Loss:", loss_loc)
    loss.backward()
    overfit_optimizer.step()
    overfit_optimizer.zero_grad()
    overfit_warmup.step()

    all_losses.append((loss.item(), loss_obj.item(), loss_class.item(), loss_loc.item()))

    print(f"Loss in epoch {epoch+1}:", loss.item())



In [ ]:
# plot loss over the epochs

plt.figure(figsize=(10, 5))
labels = ["Total Loss", "Objectness Loss", "Classification Loss", "Localization Loss"]
for l, lbls in zip(zip(*all_losses), labels):
    plt.plot(l, label=lbls)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Train vs Test Loss over Epochs")
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
def get_coords_from_pred(pred, threshold, image_size):
    class_maps = torch.softmax(pred[:, 5:], dim=1)
    max_probs, class_ids = torch.max(class_maps, dim=1)

    obj_map = torch.sigmoid(pred[:, 0])
    cell_confs = obj_map * max_probs # to get the total confidence for each cell, multiply the cell confidence with the class probability
    obj_mask = cell_confs > threshold
    obj_indices = torch.nonzero(obj_mask, as_tuple=True)
    # obj_indices list of length 3, containing tensors each with as many elements as there are cells in the batch with (objectness_conf * class_conf) > threshold
    # first element are the batch indices, second element the y-coordinates (row) and third element the x-coordinates (column) of cells with (objectness_conf * class_conf) > threshold

    obj_classes = class_ids[obj_indices] # because obj_indices is a tuple (batch_idx_tensor, y_coord_tensor, x_coord_tensor) we can index like class_ids[batch_idx_tensor, y_coord_tensor, x_coord_tensor]
    obj_conf = cell_confs[obj_indices] 

    loc_maps = torch.sigmoid(pred[:, 1:5])
    bbox_preds = yolo_to_bbox(loc_maps, img_size=image_size, map_size=pred.shape[-1])
    obj_bboxes = bbox_preds[obj_indices]

    return torch.stack(obj_indices, dim=1), obj_classes, obj_conf, obj_bboxes


In [ ]:
def get_bbox_preds(batch_size, pred, threshold=0.25, image_size=image_size, nms_treshold=0.5):
    obj_indices, classes, confidences, bboxes = get_coords_from_pred(pred, threshold=threshold, image_size=image_size)

    nms_classes = []
    nms_bboxes = []
    nms_confidences = []

    for i in range(batch_size):
        # draw the boxes around the correct objects
        pred_mask = obj_indices[:, 0] == i
        obj_conf = confidences[pred_mask] # confidence scores of all predictions above the threshold for the current image
        obj_bboxes = bboxes[pred_mask]  # bbox predictions for object predictions above the threshold for the current image

        nms_indices = nms(obj_bboxes, obj_conf, iou_threshold=nms_treshold) # return the indices of the bbox-tensor (pred_loc) that are kept after nms
        nms_preds_indices = torch.nonzero(pred_mask)[nms_indices] # because the indices are relative to the pred_loc tensor, we calculate the absolute indices of the predictions (to index the ouput of get_coords_from_pred())

        nms_pred_classes = classes[nms_preds_indices] # all relevant classes after nms
        nms_pred_bboxes = obj_bboxes[nms_indices] # filter the bbox predictions to only keep the ones that are kept after nms
        nms_pred_conf = obj_conf[nms_indices]

        nms_classes.append(nms_pred_classes)
        nms_bboxes.append(nms_pred_bboxes)
        nms_confidences.append(nms_pred_conf)

    return nms_classes, nms_bboxes, nms_confidences

In [ ]:
def get_box_coords(box, image_size):
    class_id, x_c, y_c, w, h = box
    width = w * image_size
    height = h * image_size
    x1 = x_c * image_size - width / 2
    y1 = y_c * image_size - height / 2
    return (x1, y1, width, height)

In [ ]:
def visualize_bbox_preds(X, pred_classes, pred_bboxes, confidences, pre_mean, pre_std, y=None, image_size=image_size):
    fig, axs = plt.subplots(4, 4, figsize=(15, 12))
    axs = axs.flatten()

    mean = torch.tensor(pre_mean).view(3, 1, 1) # because we have a tensor with shape [3, 448, 448], to process each channel seperately, we have to create a tensor with shape [3, 1, 1]
    std  = torch.tensor(pre_std).view(3, 1, 1)

    # visualize the first 16 images in the batch
    for i in range(16):
        # show the image with the classified label
        # X[i] is the tensor of the normalized image (with mean and std defined above) => with those parameters, we convert back to the normal image
        normal_image = X[i] * std + mean # revert the normalization process by multiplying with the standard deviation and adding the mean
        
        ax = axs[i]
        ax.imshow(normal_image.permute(1, 2, 0))
        ax.axis("off")
        
        
        # iterate over every prediction for the current image, get the coordinates and draw them into the current image
        for (cls, loc, conf) in zip(pred_classes[i], pred_bboxes[i], confidences[i]):
            x1, y1, x2, y2 = loc.cpu().detach().numpy()
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=2))
            pred_class_idx = cls.item()
            ax.annotate(f"{class_names[pred_class_idx]} - {conf.item() * 100:.2f}%", 
                        xy=(x1, y1 - image_size * 0.02), 
                        color='black',
                        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'), 
                        fontsize=8)
        # correct prediction in green (if available)
        if y:
            x1_gt, y1_gt, w_gt, h_gt = get_box_coords(y[i][0], image_size)
            ax.add_patch(plt.Rectangle((x1_gt, y1_gt), w_gt, h_gt, fill=False, edgecolor='green', linewidth=2))

        # => torchvision.util.draw_bounding_boxes() can also draw bounding boxes in a given image

    plt.tight_layout()
    plt.show()


In [ ]:
# compare the predicted bounding boxes with the GT bounding boxes
pred = model(X)
pred_classes, pred_bboxes, pred_confidences = get_bbox_preds(batch_size=X.shape[0], pred=pred, threshold=0.25, image_size=image_size, nms_treshold=0.5)
visualize_bbox_preds(X.to('cpu'), pred_classes, pred_bboxes, pred_confidences, mean, std, y=y, image_size=X.shape[-1])